# Module 05 — Hash Tables and Collision Resolution

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_group_anagrams import group_anagrams
from p02_top_k_frequent import top_k_frequent
from p03_longest_consecutive import longest_consecutive

print("module 05: Hash Tables and Collision Resolution")
print("problems available:", 8)
for name in ['p01_group_anagrams', 'p02_top_k_frequent', 'p03_longest_consecutive', 'p04_first_unique_char', 'p05_contains_nearby_duplicate', 'p06_is_isomorphic', 'p07_subarrays_div_by_k', 'p08_four_sum_count']:
    print(f"  {name}")

## 1. Baseline — `p01_group_anagrams`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
got = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert got == [["bat"], ["eat", "tea", "ate"], ["tan", "nat"]]
assert group_anagrams([]) == []
assert group_anagrams([""]) == [[""]]
assert group_anagrams(["a"]) == [["a"]]
# No anagrams at all.
assert group_anagrams(["abc", "def"]) == [["abc"], ["def"]]
# Identical words group together.
assert group_anagrams(["ab", "ab"]) == [["ab", "ab"]]
# Same letters, different multiplicities are NOT anagrams.
assert group_anagrams(["aab", "abb"]) == [["aab"], ["abb"]]
# Every input word appears exactly once across the groups.
words = ["listen", "silent", "enlist", "google", "banana", "elgoog"]
groups = group_anagrams(words)
flat = [w for g in groups for w in g]
assert sorted(flat) == sorted(words)
assert len(groups) == 3

print("all assertions held")

## 2. Predict before you run

You insert 100,000 consecutive integers and ask for the longest consecutive run. Predict the number of set-membership tests performed with, and without, the 'only start from a run start' guard. One of those numbers is about 100,000 and the other is about 5,000,000,000.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]
assert top_k_frequent([1], 1) == [1]
# Ties broken by ascending value.
assert top_k_frequent([1, 2, 3], 3) == [1, 2, 3]
assert top_k_frequent([3, 2, 1], 2) == [1, 2]
# All the same element.
assert top_k_frequent([5, 5, 5], 1) == [5]
# Negative values.
assert top_k_frequent([-1, -1, 2], 1) == [-1]
# k must be validated, not silently clamped.
with pytest.raises(ValueError):
    top_k_frequent([1, 2], 3)
# Ordering is strictly by descending frequency.
data = [4, 4, 4, 4, 7, 7, 7, 9, 9, 1]
assert top_k_frequent(data, 3) == [4, 7, 9]
assert top_k_frequent(data, 4) == [4, 7, 9, 1]
# O(n): a sort-everything solution is fine here, but this checks scale.
big = [i % 1000 for i in range(100_000)]
assert len(top_k_frequent(big, 10)) == 10

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive([]) == 0
assert longest_consecutive([1]) == 1
# Duplicates must not inflate the length.
assert longest_consecutive([1, 1, 1]) == 1
assert longest_consecutive([1, 2, 2, 3]) == 3
# No consecutive pair.
assert longest_consecutive([10, 20, 30]) == 1
# Negative and mixed signs.
assert longest_consecutive([-3, -2, -1, 0, 1]) == 5
assert longest_consecutive([-1, 1]) == 1
# A long single run - this is where the run-start check earns its, # keep. Without it this input is quadratic.
assert longest_consecutive(list(range(100_000))) == 100_000
# And a reversed one, to be sure order does not matter.
assert longest_consecutive(list(range(50_000, 0, -1))) == 50_000

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. A hash map answers 'have I seen this?' in O(1) and nothing about order or range.
2. A canonical key must be equal exactly when two inputs are equivalent - no more, no less.
3. Walking only from run starts is what turns an O(n^2) scan into O(n).

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem